<a href="https://colab.research.google.com/github/MokshBuddhadev/credit-risk-assessment/blob/main/02_modeling_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Risk Assessment — Final Modeling Notebook
**Dataset:** UCI Default of Credit Card Clients (Yeh, 2016) — same as reference paper  
**Goal:** Outperform Bhandary & Ghosh (2025) on F1, AUC, G-Mean, and Sensitivity

## CELL 1 — Install & Import All Libraries

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────
# Run this once; restart runtime if prompted
!pip install optuna xgboost lightgbm catboost shap imbalanced-learn --quiet

# ── Standard library imports ───────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Sklearn: preprocessing, model selection, metrics ──────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)

# ── Sklearn: models ────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    StackingClassifier
)

# ── Boosting libraries ─────────────────────────────────────────────────
from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier

# ── Imbalanced learning ────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE

# ── Deep Learning ──────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# ── Hyperparameter Optimisation ────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Explainability ─────────────────────────────────────────────────────
import shap

print('All libraries loaded successfully.')
print('TensorFlow version:', tf.__version__)

## CELL 2 — Load Dataset & Feature Engineering

**Why this matches the paper:**  
Same UCI dataset (Yeh, 2016), 30,000 records, same 23 original features, same target variable.  
ID column is dropped — it is a row index with no predictive value.

**What we add (paper did NOT do this):**  
8 engineered features that capture payment behaviour patterns. These are derived from existing columns — no external data.

In [ ]:
# ── Load dataset (same source as paper) ───────────────────────────────
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"
df = pd.read_excel(URL, header=1)
df = df.rename(columns={'default payment next month': 'default'})
df = df.drop(columns=['ID'])  # ID is a row index — no predictive value

print('Dataset shape (raw):', df.shape)
print('Default rate:', round(df['default'].mean(), 4), '(', df['default'].sum(), 'defaults out of', len(df), ')')
print('No duplicates:', df.duplicated().sum() == 0)
print('No missing values:', df.isnull().sum().sum() == 0)

# ── Feature Engineering ────────────────────────────────────────────────
# All features are derived ONLY from existing columns — no data leakage.
# Feature engineering is done BEFORE the train/test split so that
# derived columns are available on both splits. This is safe because
# we are not computing any statistics (mean/std) across the full dataset;
# we are only doing arithmetic on individual rows.

pay_cols    = ['PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']
bill_cols   = ['BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6']
status_cols = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']

# FE-1: Total payment and bill over 6 months (matches paper's PaySum/BillSum)
df['PAY_SUM']    = df[pay_cols].sum(axis=1)
df['BILL_SUM']   = df[bill_cols].sum(axis=1)

# FE-2: Credit utilisation rate — how much of the limit is being billed
df['UTIL_RATE']  = df['BILL_SUM'] / (df['LIMIT_BAL'] + 1)

# FE-3: Payment ratio — proportion of total bill actually paid
df['PAY_RATIO']  = df['PAY_SUM'] / (df['BILL_SUM'] + 1)

# FE-4: Number of months with payment delay (PAY_X > 0 means delayed)
df['DELAY_COUNT']= (df[status_cols] > 0).sum(axis=1)

# FE-5: Worst (maximum) single-month delay
df['MAX_DELAY']  = df[status_cols].max(axis=1)

# FE-6: Average monthly bill amount
df['AVG_BILL']   = df[bill_cols].mean(axis=1)

# FE-7: Bill trend — rising bill is a risk signal
df['BILL_TREND'] = df['BILL_AMT1'] - df['BILL_AMT6']

# FE-8: Payment consistency — high std means erratic payment behaviour
df['PAY_STD']    = df[pay_cols].std(axis=1)

print('\nDataset shape after feature engineering:', df.shape)
print('New features added:', 8)

## CELL 3 — Train/Test Split → Scale → SMOTE

**Correct pipeline order (critical for academic integrity):**
1. Split into train/test first (test set is never touched again)
2. Fit scaler on train only → transform both train and test
3. Apply SMOTE only on the scaled training set
4. Create a validation split from training data for early stopping

**Why this matters:**  
- Fitting the scaler on the full dataset (or re-fitting on SMOTE output) leaks test statistics into training. Your old code did both.
- SMOTE must run AFTER scaling so interpolation between samples is done in a normalised feature space. Your old code ran SMOTE before scaling.
- The test set stays imbalanced (22% default rate, same as reality) — exactly how the paper evaluated its models.

In [ ]:
# ── Step 1: Separate features and target ──────────────────────────────
X = df.drop(columns=['default'])
y = df['default']

feature_names = X.columns.tolist()  # save for SHAP later

# ── Step 2: Train / Test split (80/20, stratified) ────────────────────
# stratify=y ensures both splits have the same 22% default rate
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ── Step 3: Scale — fit ONLY on train, apply to both ──────────────────
# NEVER call fit_transform on test data or on SMOTE-augmented data.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit here, transform here
X_test_sc  = scaler.transform(X_test)         # only transform, no fit

# ── Step 4: Validation split from training data ───────────────────────
# Used for early stopping in LightGBM and DNN.
# This keeps the test set completely blind during training.
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_sc, y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train
)

# ── Step 5: SMOTE on training split only ─────────────────────────────
# SMOTE interpolates synthetic minority samples in scaled feature space.
# Applying it after scaling prevents high-magnitude features from
# dominating the KNN interpolation inside SMOTE.
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_tr, y_tr)

print('Train size (original):', X_train_sc.shape[0])
print('Train (after SMOTE):  ', X_train_res.shape[0], '| Class dist:', dict(zip(*np.unique(y_train_res, return_counts=True))))
print('Validation size:      ', X_val.shape[0])
print('Test size:            ', X_test_sc.shape[0], '| Default rate:', round(y_test.mean(), 4))
print('\nTest set is NEVER seen during training or threshold tuning.')

## CELL 4 — Evaluation Function

Computes all 7 metrics the paper reports: Accuracy, Sensitivity (Recall), Specificity, Precision, F1, G-Mean, AUC. Results are printed side-by-side with the paper's DNN for direct comparison.

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    """
    Computes all metrics reported in Bhandary & Ghosh (2025).
    G-Mean = sqrt(Sensitivity * Specificity) — the primary metric
    for imbalanced classification.
    """
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred)            # Sensitivity
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_prob)
    spec = recall_score(y_true, y_pred, pos_label=0)  # Specificity
    gmn  = np.sqrt(rec * spec)

    # Paper's best model (DNN) reference values
    ref  = dict(acc=0.8180, f1=0.4820, auc=0.77, gmn=0.6027, rec=0.3869, spec=0.9388, prec=0.6390)

    def flag(val, ref_val):
        return 'BEATS ✓' if val > ref_val else ('TIES' if abs(val - ref_val) < 0.0001 else 'behind')

    print(f"\n{'='*60}")
    print(f"  MODEL: {name}")
    print(f"{'='*60}")
    print(f"  {'Metric':<16} {'Yours':>8}   {'Paper DNN':>10}   {'Status':>12}")
    print(f"  {'-'*54}")
    print(f"  {'Accuracy':<16} {acc:>8.4f}   {ref['acc']:>10.4f}   {flag(acc, ref['acc']):>12}")
    print(f"  {'F1 Score':<16} {f1:>8.4f}   {ref['f1']:>10.4f}   {flag(f1, ref['f1']):>12}")
    print(f"  {'AUC':<16} {auc:>8.4f}   {ref['auc']:>10.4f}   {flag(auc, ref['auc']):>12}")
    print(f"  {'G-Mean':<16} {gmn:>8.4f}   {ref['gmn']:>10.4f}   {flag(gmn, ref['gmn']):>12}")
    print(f"  {'Sensitivity':<16} {rec:>8.4f}   {ref['rec']:>10.4f}   {flag(rec, ref['rec']):>12}")
    print(f"  {'Specificity':<16} {spec:>8.4f}   {ref['spec']:>10.4f}   {flag(spec, ref['spec']):>12}")
    print(f"  {'Precision':<16} {prec:>8.4f}   {ref['prec']:>10.4f}   {flag(prec, ref['prec']):>12}")

    return {'Model': name, 'Accuracy': acc, 'Precision': prec,
            'Sensitivity': rec, 'Specificity': spec,
            'F1': f1, 'AUC': auc, 'G-Mean': gmn}

print('evaluate_model() function defined.')

## CELL 5 — Tuned XGBoost with Optuna

**Fix from old code:** `use_label_encoder=False` is removed (deprecated since XGBoost 1.6).  
**Why Optuna:** The paper used default hyperparameters. 50-trial Bayesian search finds a significantly better configuration.  
**No leakage:** `cross_val_score` runs entirely on `X_train_res` / `y_train_res` — test set never seen.

In [ ]:
def xgb_objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 300, 1000),
        'max_depth'       : trial.suggest_int('max_depth', 3, 9),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'gamma'           : trial.suggest_float('gamma', 0.0, 2.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-4, 5.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-4, 5.0, log=True),
        'eval_metric'     : 'auc',
        'random_state'    : 42,
        'tree_method'     : 'hist',   # faster on GPU/CPU both
        'device'          : 'cpu',
    }
    model = XGBClassifier(**params)
    # CV is on SMOTE-augmented training data — test set not involved
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(
        model, X_train_res, y_train_res,
        cv=cv, scoring='roc_auc', n_jobs=-1
    ).mean()
    return score

print('Running Optuna search for XGBoost (50 trials)...')
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print('\nBest XGBoost AUC (CV):', round(study_xgb.best_value, 4))
print('Best params:', study_xgb.best_params)

# Train final XGBoost on full SMOTE training set
best_xgb = XGBClassifier(
    **study_xgb.best_params,
    eval_metric='auc',
    random_state=42,
    tree_method='hist',
    device='cpu'
)
best_xgb.fit(X_train_res, y_train_res)

xgb_pred = best_xgb.predict(X_test_sc)
xgb_prob = best_xgb.predict_proba(X_test_sc)[:, 1]
xgb_results = evaluate_model('Tuned XGBoost', y_test, xgb_pred, xgb_prob)

## CELL 6 — LightGBM

**Fix from old code:** Early stopping now monitors a held-out VALIDATION set carved from training data, not the test set.  
The validation set (`X_val`, `y_val`) was created in Cell 3 from the training data before SMOTE. This ensures zero leakage.

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators    = 1000,
    learning_rate   = 0.03,
    num_leaves      = 63,
    max_depth       = -1,
    min_child_samples = 20,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    reg_alpha       = 0.1,
    reg_lambda      = 0.1,
    random_state    = 42,
    n_jobs          = -1,
    verbose         = -1
)

# Early stopping on VALIDATION set — NOT on test set
# X_val and y_val come from Cell 3 (split from training data only)
lgb_model.fit(
    X_train_res, y_train_res,
    eval_set        = [(X_val, y_val)],
    callbacks       = [
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=-1)
    ]
)

print(f'Best iteration: {lgb_model.best_iteration_}')

lgb_pred = lgb_model.predict(X_test_sc)
lgb_prob = lgb_model.predict_proba(X_test_sc)[:, 1]
lgb_results = evaluate_model('LightGBM', y_test, lgb_pred, lgb_prob)

## CELL 7 — CatBoost

CatBoost often matches or exceeds XGBoost/LightGBM on tabular data with less tuning. It handles imbalanced data well and is robust to overfitting. The paper did not include CatBoost.

In [ ]:
cat_model = CatBoostClassifier(
    iterations      = 1000,
    learning_rate   = 0.03,
    depth           = 6,
    l2_leaf_reg     = 3,
    subsample       = 0.8,
    colsample_bylevel = 0.8,
    random_seed     = 42,
    eval_metric     = 'AUC',
    early_stopping_rounds = 50,
    verbose         = 0
)

# Early stopping on VALIDATION set — NOT on test set
cat_model.fit(
    X_train_res, y_train_res,
    eval_set        = (X_val, y_val)
)

cat_pred = cat_model.predict(X_test_sc)
cat_prob = cat_model.predict_proba(X_test_sc)[:, 1]
cat_results = evaluate_model('CatBoost', y_test, cat_pred, cat_prob)

## CELL 8 — Advanced Deep Neural Network

**Improvements over paper's DNN:**  
- Paper: 4 hidden layers, ReLU, 25 epochs, no regularisation  
- Ours: 4 hidden layers + BatchNorm + Dropout at each layer, up to 200 epochs with EarlyStopping, ReduceLROnPlateau

**Fix from old code:** EarlyStopping now monitors `val_auc` on the VALIDATION set (from Cell 3), not the test set.

In [ ]:
def build_advanced_dnn(input_dim):
    """
    Architecture improvements over paper:
    - BatchNormalization after each dense layer (stabilises training)
    - Dropout (prevents overfitting)
    - Adam with learning rate scheduling
    """
    model = Sequential([
        Dense(256, activation='relu', input_dim=input_dim),
        BatchNormalization(),
        Dropout(0.4),

        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(32, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),

        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer = Adam(learning_rate=0.001),
        loss      = 'binary_crossentropy',
        metrics   = ['AUC']
    )
    return model

tf.random.set_seed(42)
dnn = build_advanced_dnn(X_train_res.shape[1])

callbacks = [
    # Monitor val_auc on VALIDATION set (from Cell 3) — not test set
    EarlyStopping(
        monitor='val_auc', patience=20,
        restore_best_weights=True, mode='max'
    ),
    ReduceLROnPlateau(
        monitor='val_auc', patience=8,
        factor=0.5, mode='max', min_lr=1e-5
    )
]

history = dnn.fit(
    X_train_res, y_train_res,
    validation_data = (X_val, y_val),   # validation set, NOT test set
    epochs          = 200,
    batch_size      = 256,
    callbacks       = callbacks,
    verbose         = 1
)

dnn_prob = dnn.predict(X_test_sc, verbose=0).ravel()
dnn_pred = (dnn_prob >= 0.5).astype(int)
dnn_results = evaluate_model('Advanced DNN', y_test, dnn_pred, dnn_prob)

## CELL 9 — Stacking Ensemble (XGBoost + LightGBM + CatBoost + RF)

Stacking combines predictions from multiple diverse models via a meta-learner.  
`cv=5` means each base model generates out-of-fold predictions on training data, which the meta-learner (Logistic Regression) then learns from. No test set contact.  
The paper did not use ensembling.

In [ ]:
# Base learners — use tuned XGBoost params + fixed LightGBM/CatBoost/RF
base_learners = [
    ('xgb', XGBClassifier(
        **study_xgb.best_params,
        eval_metric='auc', random_state=42,
        tree_method='hist', device='cpu'
    )),
    ('lgb', lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05,
        num_leaves=63, random_state=42,
        n_jobs=-1, verbose=-1
    )),
    ('cat', CatBoostClassifier(
        iterations=500, learning_rate=0.05,
        depth=6, random_seed=42, verbose=0
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300, max_depth=10,
        random_state=42, n_jobs=-1
    ))
]

# Meta-learner: Logistic Regression on out-of-fold predictions
stack_model = StackingClassifier(
    estimators     = base_learners,
    final_estimator= LogisticRegression(C=1.0, max_iter=1000),
    cv             = 5,
    stack_method   = 'predict_proba',
    n_jobs         = -1
)

print('Training stacking ensemble (this takes a few minutes)...')
stack_model.fit(X_train_res, y_train_res)

stack_pred = stack_model.predict(X_test_sc)
stack_prob = stack_model.predict_proba(X_test_sc)[:, 1]
stack_results = evaluate_model('Stacking Ensemble', y_test, stack_pred, stack_prob)

## CELL 10 — Threshold Optimisation (on Validation Set)

**Fix from old code:** The old code optimised the threshold directly on `y_test` — this is data leakage because the optimal threshold was tuned to the specific test distribution.  

**Correct approach:** Find the optimal threshold using the VALIDATION set predictions, then apply it to the test set. The test set sees this threshold for the first time at evaluation.

In [ ]:
def find_optimal_threshold(model, X_val, y_val, metric='f1'):
    """
    Searches thresholds on the VALIDATION set.
    Returns the threshold, not the test score.
    This threshold is then applied blindly to the test set.
    """
    val_prob = model.predict_proba(X_val)[:, 1]
    thresholds = np.arange(0.20, 0.80, 0.005)
    best_score, best_t = -1, 0.5

    for t in thresholds:
        preds = (val_prob >= t).astype(int)
        if metric == 'f1':
            score = f1_score(y_val, preds, zero_division=0)
        elif metric == 'gmean':
            rec  = recall_score(y_val, preds)
            spec = recall_score(y_val, preds, pos_label=0)
            score = np.sqrt(rec * spec)
        else:
            score = f1_score(y_val, preds, zero_division=0)

        if score > best_score:
            best_score, best_t = score, t

    print(f'  Optimal threshold ({metric}) on VAL set: {best_t:.3f} | Val {metric}: {best_score:.4f}')
    return best_t

# Find optimal F1 threshold for each model using VALIDATION set
print('--- Finding optimal thresholds on VALIDATION set ---')
print('XGBoost:')
t_xgb   = find_optimal_threshold(best_xgb,   X_val, y_val, metric='f1')
print('LightGBM:')
t_lgb   = find_optimal_threshold(lgb_model,  X_val, y_val, metric='f1')
print('CatBoost:')
t_cat   = find_optimal_threshold(cat_model,  X_val, y_val, metric='f1')
print('Stacking:')
t_stack = find_optimal_threshold(stack_model, X_val, y_val, metric='f1')

print('\n--- Applying thresholds to TEST set (first time test set is seen) ---')

xgb_pred_opt   = (xgb_prob   >= t_xgb).astype(int)
lgb_pred_opt   = (lgb_prob   >= t_lgb).astype(int)
cat_pred_opt   = (cat_prob   >= t_cat).astype(int)
stack_pred_opt = (stack_prob >= t_stack).astype(int)

xgb_opt_results   = evaluate_model('XGBoost (Tuned Threshold)',   y_test, xgb_pred_opt,   xgb_prob)
lgb_opt_results   = evaluate_model('LightGBM (Tuned Threshold)',  y_test, lgb_pred_opt,   lgb_prob)
cat_opt_results   = evaluate_model('CatBoost (Tuned Threshold)',  y_test, cat_pred_opt,   cat_prob)
stack_opt_results = evaluate_model('Stacking (Tuned Threshold)',  y_test, stack_pred_opt, stack_prob)

## CELL 11 — Final Comparison Table

All models vs the reference paper's best model (DNN). Results are computed on the same imbalanced test set using the same 7 metrics the paper reports.

In [ ]:
# Collect all results
all_results = [
    xgb_results,
    lgb_results,
    cat_results,
    dnn_results,
    stack_results,
    xgb_opt_results,
    lgb_opt_results,
    cat_opt_results,
    stack_opt_results,
]

results_df = pd.DataFrame(all_results).set_index('Model')

# Paper's reference values (Table 5 in Bhandary & Ghosh, 2025)
paper_dnn = {
    'Accuracy': 0.8180, 'Precision': 0.6390,
    'Sensitivity': 0.3869, 'Specificity': 0.9388,
    'F1': 0.4820, 'AUC': 0.7700, 'G-Mean': 0.6027
}
results_df.loc['Paper DNN (Reference — Bhandary & Ghosh, 2025)'] = paper_dnn

print('\n' + '='*90)
print('FINAL COMPARISON vs REFERENCE PAPER (Bhandary & Ghosh, 2025)')
print('='*90)
print(results_df.round(4).to_string())

# Find overall best model by AUC
models_only = results_df.drop('Paper DNN (Reference — Bhandary & Ghosh, 2025)')
best_name   = models_only['AUC'].idxmax()
best_row    = models_only.loc[best_name]

print('\n' + '='*60)
print(f'BEST MODEL: {best_name}')
print('='*60)
metrics = ['Accuracy', 'F1', 'AUC', 'G-Mean', 'Sensitivity']
beat_count = 0
for m in metrics:
    ref = paper_dnn[m]
    val = best_row[m]
    diff = val - ref
    status = 'BEATS ✓' if diff > 0.0001 else ('TIES' if abs(diff) < 0.0001 else 'behind')
    if diff > 0.0001:
        beat_count += 1
    print(f'  {m:<14}: {val:.4f} vs {ref:.4f}  ({diff:+.4f})  [{status}]')

print(f'\nBeats paper on {beat_count}/{len(metrics)} primary metrics.')

## CELL 12 — ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

plot_models = [
    ('Tuned XGBoost',    xgb_prob,   '#1f77b4'),
    ('LightGBM',         lgb_prob,   '#2ca02c'),
    ('CatBoost',         cat_prob,   '#d62728'),
    ('Advanced DNN',     dnn_prob,   '#9467bd'),
    ('Stacking Ensemble',stack_prob, '#ff7f0e'),
]

for label, prob, color in plot_models:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc_val:.4f})', color=color, linewidth=2)

# Paper's reference AUC (DNN = 0.77) — shown as a reference point
ax.axhline(y=0, xmin=0, xmax=0, color='gray', linestyle='--', label='Paper DNN AUC = 0.77 (reference)')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models vs Reference Paper', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## CELL 13 — SHAP Feature Importance

**Fix from old code:** `X_test_sc` was a raw numpy array, so SHAP could not resolve feature names — resulting in a blank plot.  
**Fix:** Convert `X_test_sc` to a DataFrame with column names before passing to SHAP.

In [ ]:
# Convert numpy array to DataFrame so SHAP can read feature names
X_test_sc_df = pd.DataFrame(X_test_sc, columns=feature_names)

# Use a sample of 500 for speed (SHAP TreeExplainer is exact, not approximate)
sample_df = X_test_sc_df.iloc[:500]

explainer   = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(sample_df)

# Summary bar plot — mean |SHAP| per feature
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    sample_df,
    plot_type='bar',
    max_display=20,
    show=False
)
plt.title('SHAP Feature Importance — Tuned XGBoost', fontsize=13)
plt.tight_layout()
plt.show()

# Beeswarm plot — direction and magnitude of each feature's impact
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    sample_df,
    plot_type='dot',
    max_display=15,
    show=False
)
plt.title('SHAP Beeswarm — Feature Impact Direction', fontsize=13)
plt.tight_layout()
plt.show()

## CELL 14 — Confusion Matrices for Best Models

In [ ]:
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['No Default', 'Default'],
        yticklabels=['No Default', 'Default'],
        ax=ax
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_confusion(y_test, stack_pred_opt, 'Stacking (Tuned Threshold)')
plot_confusion(y_test, xgb_pred_opt,   'XGBoost (Tuned Threshold)')
plot_confusion(y_test, lgb_pred_opt,   'LightGBM (Tuned Threshold)')

## CELL 15 — Unsupervised: KMeans Clustering + PCA Visualisation

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Cluster on the training scaled data only — no test leakage
kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(X_train_sc)

sil = silhouette_score(X_train_sc, cluster_labels)
print(f'Silhouette Score (KMeans, k=2): {sil:.4f}')

# PCA for 2D visualisation
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train_sc)

plt.figure(figsize=(8, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels,
            cmap='viridis', alpha=0.4, s=5)
plt.title('KMeans Clusters (PCA 2D projection)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.colorbar(label='Cluster')
plt.tight_layout()
plt.show()

print(f'Variance explained by 2 PCA components: {pca.explained_variance_ratio_.sum():.2%}')